# 📔 集成学习 (Ensemble Learning) 与 Boosting 笔记

## 1. 集成学习基础 
> 核心思想：组合多个“弱学习器”(Weak Learners) 来构建一个“强学习器”(Strong Learner)。

### 💡 三大主流流派：
1. **Bagging (并行)**: 
   - 方式：独立训练多个模型，结果取平均或投票。
   - 目标：降低 **方差 (Variance)**，防止过拟合。
   - 代表：随机森林 (Random Forest)。
2. **Boosting (串行)**: 
   - 方式：模型按顺序训练，后一个模型尝试修复前一个模型的错误。
   - 目标：降低 **偏差 (Bias)**，提高准确度。
   - 代表：AdaBoost, GBDT, XGBoost, LightGBM。
3. **Stacking (层叠)**: 
   - 方式：第一层模型的预测结果作为第二层模型的输入。
   - 目标：结合不同类型算法的优势。

---

## 2. Boosting 核心原理
> 逻辑：步步为营，错题重做。

- **工作流程**:
  Step 1: 训练模型 M1，计算预测误差。
  Step 2: 根据 M1 的误差，调整数据权重或计算残差。
  Step 3: 训练模型 M2 去拟合这些误差。
  Step 4: 循环迭代，最终结果 = M1 + M2 + ... + Mn。

- **两大演化路径**:
  - **AdaBoost**: 提高被分错样本的权重，让下个模型“重点关注”难点。
  - **Gradient Boosting (GBDT)**: 通过负梯度来拟合残差，让下个模型去“填补”之前的差距。

---

## 3. XGBoost 深度解析 (The Extreme Gradient Boosting)
> 定义：基于 GBDT 的大规模并行梯度提升算法，是目前最强、应用最广的 Boosting 实现。

### 🚀 核心进化点 (为什么它强？)
1. **数学优化 (精度高)**:
   - 传统 GBDT 只用一阶导数（梯度）。
   - XGBoost 使用 **二阶泰勒展开** (一阶导 $g$ + 二阶导 $h$)，对目标函数的拟合更精准。
2. **正则化 (防止过拟合)**:
   - 在目标函数中加入了 **L1/L2 正则项**。
   - 控制树的复杂度（叶子节点数、权重大小），泛化能力更强。
3. **工程优化 (速度快)**:
   - **特征并行**: 预排序 (Pre-sorting) 后，在寻找分裂点时可以并行计算。
   - **稀疏感知**: 自动学习缺失值的默认分裂方向。
   - **近似算法**: 对大数据量采用分位数直方图，不再遍历所有点。

---

## 4. 对比：Bagging vs Boosting (面试高频)

| 维度 | Bagging (随机森林) | Boosting (XGBoost) |
| :--- | :--- | :--- |
| **构建方式** | 并行 (各棵树独立) | 串行 (树之间有依赖) |
| **数据使用** | 有放回随机抽样 (Bootstrap) | 全量数据 (或带权重的样本) |
| **主要目标** | 减小方差 (降低模型波动) | 减小偏差 (提升预测精度) |
| **对异常值** | 较鲁棒 (不敏感) | 较敏感 (容易去死磕噪声导致过拟合) |
| **模型复杂度** | 树通常很深 (追求低偏差) | 树通常很浅 (通过叠加来变强) |

---

## 5. 算法选择建议
- **数据有缺失值、离散值多、追求极致精度** ➔ 首选 **XGBoost / LightGBM**。
- **数据样本量小、担心模型不稳、没时间调参** ➔ 首选 **Random Forest**。
- **不同模型（如LR, SVM, RF）表现各异** ➔ 尝试 **Stacking** 进行融合。

---

In [22]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# 1. 构造模拟数据集
data = {
    '年龄': [25, 45, 35, 50, 23, 40, 60, 48, 22, 33],       # 数值特征
    '年收入': [50000, 80000, 65000, 120000, 45000, 70000, 95000, 110000, 48000, 55000], # 数值特征
    '城市': ['北京', '上海', '北京', '深圳', '成都', '上海', '深圳', '成都', '北京', '深圳'], # 类别特征
    '性别': ['男', '女', '女', '男', '女', '男', '男', '女', '男', '女'], # 类别特征
    '是否流失': [0, 1, 0, 1, 0, 0, 1, 1, 0, 0]             # 标签 (Target)
}

df = pd.DataFrame(data)

# 2. 数据预处理：处理类别特征
# 注意：XGBoost 原生支持类别特征，但需要我们将列类型显式转为 'category'
categorical_cols = ['城市', '性别']
for col in categorical_cols:
    df[col] = df[col].astype('category')

# 划分特征和标签
X = df.drop('是否流失', axis=1)
y = df['是否流失']

# 3. 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. 初始化 XGBoost 模型
# 参数解释：
# - enable_categorical: 开启原生类别特征支持
# - tree_method: 使用 'hist' (直方图算法) 是开启类别支持的前提
# - n_estimators: 树的数量
# - max_depth: 树的深度
# - learning_rate: 学习率 (步长)
# - scale_pos_weight: 处理不平衡数据 (本例数据量小，仅作演示)
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    tree_method='hist',      # 必须设为 hist 才能处理类别特征
    enable_categorical=True, # 核心：允许输入 category 类型数据
    eval_metric='logloss'
)

# 5. 训练模型
model.fit(X_train, y_train)

# 6. 模型预测
y_pred = model.predict(X_test)  #返回的是确定的类别（0 或 1）
y_prob = model.predict_proba(X_test)[:, 1] # 返回的是概率（二维数组）。第0列（预测为 0 的概率），第1列（预测为1的概率）

print("真实值:", list(y_test))
print("预测值:", list(y_pred))
# 7. 评估模型
print("--- 分类报告 ---")
print(classification_report(y_test, y_pred))
print(f"AUC 评分: {roc_auc_score(y_test, y_prob):.4f}")

真实值: [0, 1]
预测值: [np.int64(0), np.int64(0)]
--- 分类报告 ---
              precision    recall  f1-score   support

           0       0.50      1.00      0.67         1
           1       0.00      0.00      0.00         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2

AUC 评分: 0.5000


c:\Users\under\anaconda3\envs\myenv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\under\anaconda3\envs\myenv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\under\anaconda3\envs\myenv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape